# Leo Video Studio — interactive GPU rendering
Select a GPU runtime. This uses pretrained Wan, not the old shapes model. Start with one economy shot. GPU availability and sufficient memory are not guaranteed. This notebook does not host a public app.

Economy mode saves prompt embeddings with a 4-bit text encoder, unloads it, and renders shots with a separate video pipeline. Run cells in order on a fresh GPU session. Start with one shot; the integrated version still needs a real GPU test. Keep the saved project ID to resume. The 1080p export is an upscale.

In [3]:
REPO_URL = "https://github.com/mugishaleobeni/leo-video-studio.git"
assert "YOUR_USERNAME" not in REPO_URL, "Set your GitHub repository URL"
import subprocess
from pathlib import Path
if not Path("/content/leo-video-studio").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/leo-video-studio"], check=True)
%cd /content/leo-video-studio
%pip install -r requirements.txt
subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

/content/leo-video-studio
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

CompletedProcess(args=['apt-get', 'install', '-y', 'ffmpeg'], returncode=0)

Persist the project on Drive so finished scenes survive disconnections. Model cache remains temporary unless you set HF_HOME to persistent storage. Mounting Drive gives notebook code access to it.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["STUDIO_DATA"] = "/content/drive/MyDrive/leo-video-studio"
os.environ["HF_HOME"] = "/content/leo-model-cache"
os.environ["CPU_OFFLOAD"] = "1"
os.environ["LOW_MEMORY"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Tesla T4


In [4]:
from planner import make_plan
import jobs
STORY = "A father in a blue shirt embraces his daughter in a yellow coat in a sunlit garden, medium cinematic shot, natural expressions."
plan = make_plan(STORY, style="Realistic", episodes=1, shots=1, characters="Father: short dark hair, blue shirt. Daughter: yellow coat.")
import json
print(json.dumps(plan, indent=2))
# Edit plan here before proceeding.

{
  "title": "Untitled season",
  "style": "Realistic",
  "characters": "Father: short dark hair, blue shirt. Daughter: yellow coat.",
  "planner": "Draft splitter: edit for story progression; repeated text may appear",
  "episodes": [
    {
      "title": "Episode 1",
      "shots": [
        {
          "prompt": "Wide establishing shot. A father in a blue shirt embraces his daughter in a yellow coat in a sunlit garden, medium cinematic shot, natural expressions.",
          "narration": ""
        }
      ]
    }
  ]
}


In [6]:
plan["title"] = "Father and Daughter"
plan["episodes"][0]["shots"][0]["prompt"] = (
    "Medium cinematic shot of a father in a blue shirt gently "
    "embracing his daughter in a yellow coat in a sunlit garden. "
    "Natural expressions, warm light, slow camera movement."
)

In [7]:
PROJECT_ID = jobs.create(plan, "Economy 1.3B / 480p", steps=40, seed=42)
print("Save this project ID:", PROJECT_ID)
# On a later session, replace PROJECT_ID with your saved value instead of creating again.

Save this project ID: a2de5f3c8ef34787bc2eb6476dee3b9f


In [8]:
for message, outputs in jobs.render(PROJECT_ID):
    print(message, flush=True)

Loading video model. First run downloads model weights.
Preparing prompt embeddings. Economy mode unloads the text encoder before video generation.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer/spiece.model:   0%|          | 0.00/4.55M [00:00<?, ?B/s]

tokenizer/tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

...

transformer/diffusion_pytorch_model-0000(…):   0%|          | 0.00/677M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 47.98 GiB. GPU 0 has a total capacity of 14.56 GiB of which 9.99 GiB is free. Including non-PyTorch memory, this process has 4.57 GiB memory in use. Of the allocated memory 4.13 GiB is allocated by PyTorch, and 321.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [9]:
import gc
import torch
from torch.nn.attention import sdpa_kernel, SDPBackend

jobs.RENDERER.pipe.maybe_free_model_hooks()
gc.collect()
torch.cuda.empty_cache()

PROJECT_ID = "a2de5f3c8ef34787bc2eb6476dee3b9f"

with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
    for message, outputs in jobs.render(PROJECT_ID):
        print(message, flush=True)

Loading video model. First run downloads model weights.
Preparing prompt embeddings. Economy mode unloads the text encoder before video generation.
Rendering episode 1, shot 1 (1). 1 shots · about 5.1 seconds total · 1 episodes


  0%|          | 0/40 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/models/attention_dispatch.py:1814: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  out = torch.nn.functional.scaled_dot_product_attention(
/usr/local/lib/python3.13/dist-packages/diffusers/models/attention_dispatch.py:1814: UserWarning: Expected query, key and value to all be of dtype: {Half, Float}. Got Query dtype: c10::BFloat16, Key dtype: c10::BFloat16, and Value dtype: c10::BFloat16 instead. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:91.)
  out = torch.nn.functional.scaled_dot_product_attention(
/usr/local/lib/python3.13/dist-packages/diffusers/models/attention_dispatch.py:1814: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  out = torch.nn.functional.scaled_dot_product_attention(
/usr/local/

RuntimeError: No available kernel. Aborting execution.

In [10]:
from pathlib import Path
import importlib
import backend
import jobs
import torch
from torch.nn.attention import sdpa_kernel, SDPBackend

file = Path("/content/leo-video-studio/backend.py")
source = file.read_text()
old = "dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16"
new = "dtype = torch.float16"

assert old in source, "Expected dtype line not found; send me this message."
file.write_text(source.replace(old, new))

jobs.RENDERER.release()
importlib.reload(backend)
jobs.RENDERER = backend.WanRenderer()

PROJECT_ID = "a2de5f3c8ef34787bc2eb6476dee3b9f"

with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
    for message, outputs in jobs.render(PROJECT_ID):
        print(message, flush=True)

Loading video model. First run downloads model weights.
Preparing prompt embeddings. Economy mode unloads the text encoder before video generation.
Rendering episode 1, shot 1 (1). 1 shots · about 5.1 seconds total · 1 episodes


Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

Episode 1 exported at 1920×1080 (upscaled).
All episodes exported. Review shots for visual quality and character continuity.


In [11]:
from IPython.display import Video, display
from google.colab import files
video = jobs.folder(PROJECT_ID) / "episode-01.mp4"
display(Video(str(video), embed=True, width=640))
files.download(str(video))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Export is 1920×1080, upscaled from model resolution. Review quality before making a longer plan. For the browser studio, deploy app.py on a separate GPU host using README.md. This notebook has not been run on Colab in the build workspace.